[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *PyArrow*.

## ¿Qué es *Apache Arrow*?

[*Apache Arrow*](https://arrow.apache.org/) es un proyecto de código abierto que define un **formato estándar para representar datos tabulares en memoria**. Fue anunciado en 2016 por varias organizaciones del ecosistema de datos.

Uno de sus principales impulsores fue [**Wes McKinney**](https://wesmckinney.com/), creador de *Pandas*. En el artículo [*"Apache Arrow and the '10 Things I Hate About pandas'"*](https://wesmckinney.com/blog/apache-arrow-pandas-internals/) (2017) describió los problemas de fondo de *Pandas*: consumo excesivo de memoria, copias innecesarias de datos y falta de un formato común con otras herramientas. *Arrow* fue su respuesta.

El problema que resuelve es el siguiente: antes de *Arrow*, cada herramienta (*NumPy*, *Pandas*, *Spark*, bases de datos, etc.) guardaba los datos en memoria a su manera. Para pasar datos de una herramienta a otra había que **convertirlos y copiarlos**, lo que consume tiempo y memoria. *Arrow* propone que todas usen la misma representación, de modo que los datos puedan compartirse sin conversión.

### Almacenamiento por renglones y por columnas.

Una tabla puede guardarse en memoria de dos maneras:

```
Tabla:            id | producto | precio
                   1 | Cuaderno |   65.0
                   2 | Mochila  |  649.0
                   3 | Balón    |  459.0

Por renglones:    [1, Cuaderno, 65.0] [2, Mochila, 649.0] [3, Balón, 459.0]

Por columnas:     id:       [1, 2, 3]
                  producto: [Cuaderno, Mochila, Balón]
                  precio:   [65.0, 649.0, 459.0]
```

*Arrow* usa el **formato columnar**: los valores de cada columna se guardan juntos y contiguos en memoria. Esto tiene varias ventajas para el análisis de datos:

* Una operación sobre una columna (sumar precios, filtrar por producto) sólo lee los datos de esa columna.
* Los valores contiguos del mismo tipo permiten que el procesador aplique una misma instrucción a varios valores a la vez (instrucciones *SIMD*).
* Los valores de una columna se parecen entre sí, por lo que se comprimen mejor.

El almacenamiento por renglones es más conveniente cuando se leen o escriben registros completos uno por uno, como en una base de datos transaccional.

### *Zero-copy*.

Cuando dos bibliotecas entienden el mismo formato de memoria, una puede usar los datos de la otra **sin copiarlos**: ambas apuntan a la misma región de memoria. A esto se le llama *zero-copy*.

Es importante no sobreestimarlo: sólo hay *zero-copy* cuando el formato de origen y el de destino coinciden. Entre *PyArrow* y *Polars*, que usan *Arrow* internamente, casi siempre lo hay. Entre *PyArrow* y *NumPy* sólo para tipos numéricos sin valores nulos. Y convertir a un *dataframe* de *Pandas* normalmente **sí** implica una copia.

### *Arrow* y *PyArrow*.

* ***Apache Arrow*** es la especificación del formato, con implementaciones en *C++*, *Java*, *Rust*, *Go*, *R* y otros lenguajes.
* [***PyArrow***](https://arrow.apache.org/docs/python/) es la biblioteca de *Python* que implementa *Arrow*. Además del formato en memoria, incluye funciones de cálculo y lectura y escritura de *CSV*, *Parquet*, *JSON* y *Feather*.

Hoy *Arrow* está en la base de muchas herramientas del curso: *Pandas* 3 lo usa para las columnas de texto, *Polars* lo usa como formato interno y *Dask*, *DuckDB* y *Spark* lo usan para intercambiar datos.

* La siguiente celda instala *PyArrow* en el entorno actual en caso de que no esté disponible.

In [ ]:
%pip install pyarrow

* La siguiente celda importa `pyarrow` con el alias convencional `pa` y el módulo `pyarrow.compute` con el alias `pc`, junto con las demás bibliotecas que se usarán en este *notebook*.

In [ ]:
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

print(f"PyArrow {pa.__version__}")

* La siguiente celda define las rutas de los datos de ejemplo y crea el directorio `data_tmp` para los archivos que se generen.

In [ ]:
ruta_ventas = os.path.join('data', 'ventas.csv')
dir_tmp = 'data_tmp'
os.makedirs(dir_tmp, exist_ok=True)

### Los datos de ejemplo.

Los *notebooks* 34 a 37 usan los mismos datos: las ventas de una cadena de tiendas durante 2024 y 2025. Son datos sintéticos que se generan con `scripts/generar_datos_ventas.py`.

| Archivo | Contenido |
| :--- | :--- |
| `data/ventas.csv` | 20,000 ventas: fecha y hora, tienda, canal, producto, categoría, cantidad, precio unitario, descuento y correo del cliente. |
| `data/tiendas.csv` | Catálogo de 14 tiendas con ciudad, estado y región. |
| `data/tipo_cambio.csv` | Tipo de cambio diario USD/MXN en días hábiles. |

Algunos defectos son intencionales, para practicar limpieza y combinación de tablas: hay descuentos nulos, correos con mayúsculas o espacios, ventas de una tienda que no existe en el catálogo y tiendas sin ventas.

## *Arrays* de *PyArrow*.

Un *array* de *PyArrow* (`pa.Array`) es una secuencia de valores **del mismo tipo**, guardados de forma contigua en memoria. Es el equivalente de una columna.

```
pa.array(<datos>, type=<tipo>)
```

Donde:

* `<datos>` es una lista de *Python*, un arreglo de *NumPy* o cualquier otra secuencia.
* `<tipo>` es opcional y define el tipo de dato de *Arrow*, por ejemplo `pa.int32()`, `pa.float64()` o `pa.string()`. Si se omite, *PyArrow* lo infiere a partir de los datos.

Los *arrays* de *PyArrow* son **inmutables**: una vez creados no es posible modificar sus elementos. Las operaciones siempre regresan un *array* nuevo.

[Documentación de `pa.array()`](https://arrow.apache.org/docs/python/generated/pyarrow.array.html)

**Ejemplos:**

* La siguiente celda crea un *array* a partir de una lista de *Python*. *PyArrow* infiere que el tipo es `int64`.

In [ ]:
cantidades = pa.array([1, 3, 2, 5, 1])
cantidades

* La siguiente celda muestra el tipo de dato del *array* con el atributo `type`.

In [ ]:
cantidades.type

* La siguiente celda crea el mismo *array* con un tipo explícito de 8 bits, suficiente para cantidades pequeñas y que ocupa la octava parte de memoria que `int64`.

In [ ]:
cantidades_8 = pa.array([1, 3, 2, 5, 1], type=pa.int8())
print(cantidades_8.type)
print(f"int64: {cantidades.nbytes} bytes · int8: {cantidades_8.nbytes} bytes")

* La siguiente celda intentará modificar un elemento del *array*. Como los *arrays* de *PyArrow* son inmutables, se desencadenará una excepción de tipo `TypeError`.

In [ ]:
cantidades[0] = 10

### Valores nulos.

*Arrow* representa los valores faltantes como **nulos** (`null`), independientemente del tipo de dato. Para ello guarda, junto a los valores, un mapa de bits de validez (*validity bitmap*) que indica qué posiciones contienen un valor válido. Así, un *array* de enteros puede tener nulos sin convertirse a flotante, como sucede con `np.nan` en *NumPy*.

* La siguiente celda crea un *array* con un valor `None`, que *PyArrow* convierte en nulo. El tipo sigue siendo `int64`.

In [ ]:
descuentos = pa.array([0, 5, None, 10, 0])
descuentos

* La siguiente celda obtiene el número de nulos con el atributo `null_count`, que no requiere recorrer los datos, y la posición de cada uno con `is_null()`.

In [ ]:
print(f"Nulos: {descuentos.null_count}")
descuentos.is_null()

### Escalares y conversión a *Python*.

Al acceder a un elemento de un *array* se obtiene un **escalar** de *PyArrow* (`pa.Scalar`), no un objeto de *Python*. El método `as_py()` del escalar lo convierte al tipo nativo de *Python*. Para convertir el *array* completo se usa `to_pylist()`.

* La siguiente celda obtiene el primer elemento del *array* `cantidades` y muestra su tipo.

In [ ]:
primero = cantidades[0]
print(primero, type(primero))

* La siguiente celda convierte el escalar a un entero de *Python* con `as_py()`.

In [ ]:
valor = primero.as_py()
print(valor, type(valor))

* La siguiente celda convierte el *array* `descuentos` a una lista de *Python* con `to_pylist()`. Los nulos se convierten en `None`.

In [ ]:
descuentos.to_pylist()

### Conversión desde y hacia *NumPy*.

* La siguiente celda crea un *array* de *PyArrow* a partir de un arreglo de *NumPy*. Para tipos numéricos, *PyArrow* puede reutilizar la memoria del arreglo sin copiarla.

In [ ]:
precios_np = np.array([65.0, 649.0, 459.0, 120.0])
precios = pa.array(precios_np)
precios

* La siguiente celda convierte el *array* de vuelta a *NumPy* con `to_numpy(zero_copy_only=True)`, que garantiza que no se haga ninguna copia.

In [ ]:
precios.to_numpy(zero_copy_only=True)

* *NumPy* no tiene una forma de representar nulos en un arreglo de enteros. Por eso, la siguiente celda desencadenará una excepción de tipo `ArrowInvalid` al pedir la conversión sin copia de un *array* con nulos.

In [ ]:
descuentos.to_numpy(zero_copy_only=True)

* Sin la restricción, *PyArrow* hace la copia y convierte los enteros a flotantes para representar el nulo como `nan`.

In [ ]:
descuentos.to_numpy(zero_copy_only=False)

## El módulo `pyarrow.compute`.

El módulo [`pyarrow.compute`](https://arrow.apache.org/docs/python/compute.html), que se importa por convención como `pc`, contiene las funciones de cálculo sobre *arrays*. Están escritas en *C++* y aprovechan el formato columnar.

```
pc.<función>(<array>, <argumentos>)
```

Donde:

* `<función>` es el nombre de la operación.
* `<array>` es el *array* sobre el que se realiza la operación.
* `<argumentos>` son los demás argumentos de la función, por ejemplo otro *array* o un valor.

Las funciones de agregación (como `pc.sum()`) regresan un escalar; las demás regresan un *array* nuevo. Estas son las funciones que se usan en este *notebook*:

| Tipo | Funciones |
| :--- | :--- |
| Aritméticas | [`add()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.add.html), [`multiply()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.multiply.html), [`round()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.round.html) |
| Agregaciones | [`sum()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.sum.html), [`mean()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.mean.html), [`min_max()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.min_max.html), [`count()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.count.html) |
| Comparación y filtrado | [`greater()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.greater.html), [`filter()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.filter.html), [`is_in()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.is_in.html), [`if_else()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.if_else.html) |
| Texto | [`utf8_upper()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.utf8_upper.html), [`match_substring()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.match_substring.html) |
| Otras | [`cast()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.cast.html), [`fill_null()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.fill_null.html), [`value_counts()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.value_counts.html) |

La lista completa está en la [documentación de las funciones de cálculo](https://arrow.apache.org/docs/python/compute.html#available-compute-functions). En la práctica, estas operaciones suelen hacerse con *Polars* o *Pandas*, que usan *Arrow* por debajo; conocer `pyarrow.compute` es útil para trabajar directamente con datos de *Arrow* sin convertirlos.

**Ejemplos:**

* La siguiente celda define los *arrays* `precios` y `cantidades` de cinco ventas.

In [ ]:
precios = pa.array([65.0, 649.0, 459.0, 120.0, 1499.0])
cantidades = pa.array([3, 1, 2, 5, 1])

* La siguiente celda calcula el importe de cada venta multiplicando ambos *arrays* elemento a elemento con `pc.multiply()`.

In [ ]:
importes = pc.multiply(precios, cantidades)
importes

* La siguiente celda aplica un 16 % de impuesto a cada importe y redondea a dos decimales.

In [ ]:
pc.round(pc.multiply(importes, 1.16), 2)

* La siguiente celda calcula la suma y la media de los importes. El resultado de cada función es un escalar, que se convierte a *Python* con `as_py()`.

In [ ]:
print(f"Total: {pc.sum(importes).as_py()}")
print(f"Promedio: {pc.mean(importes).as_py()}")

* La siguiente celda obtiene el mínimo y el máximo en una sola pasada con `pc.min_max()`, que regresa un escalar con dos campos.

In [ ]:
pc.min_max(importes).as_py()

* La siguiente celda crea una máscara booleana con `pc.greater()` y la usa con `pc.filter()` para conservar sólo los importes mayores a 500.

In [ ]:
mascara = pc.greater(importes, 500)
pc.filter(importes, mascara)

* La siguiente celda clasifica cada importe con `pc.if_else()`: `'alto'` si es mayor a 500 y `'bajo'` en otro caso.

In [ ]:
pc.if_else(mascara, 'alto', 'bajo')

* La siguiente celda usa `pc.is_in()` para identificar qué productos pertenecen a una lista de valores.

In [ ]:
productos = pa.array(['Cuaderno', 'Mochila', 'Balón', 'Plumas', 'Audífonos'])
pc.is_in(productos, value_set=pa.array(['Cuaderno', 'Plumas', 'Mochila']))

* La siguiente celda aplica dos funciones de texto: `pc.utf8_upper()` convierte a mayúsculas y `pc.match_substring()` busca una subcadena.

In [ ]:
print(pc.utf8_upper(productos))
print(pc.match_substring(productos, 'o'))

* La siguiente celda reemplaza los nulos del *array* `descuentos` por `0` con `pc.fill_null()` y convierte el resultado a `int8` con `pc.cast()`.

In [ ]:
pc.cast(pc.fill_null(descuentos, 0), pa.int8())

* La siguiente celda cuenta las apariciones de cada valor con `pc.value_counts()`.

In [ ]:
pc.value_counts(pa.array(['tienda', 'en línea', 'tienda', 'tienda', 'en línea']))

## Tipos anidados.

Además de los tipos básicos, *Arrow* tiene tipos **anidados**, que permiten guardar datos semiestructurados (por ejemplo, provenientes de *JSON*) sin perder el formato columnar:

| Tipo | Descripción | Ejemplo de uso |
| :--- | :--- | :--- |
| [`pa.list_(<tipo>)`](https://arrow.apache.org/docs/python/generated/pyarrow.list_.html) | Lista de longitud variable con elementos del mismo tipo. | Etiquetas de un producto, artículos de un pedido. |
| [`pa.struct(<campos>)`](https://arrow.apache.org/docs/python/generated/pyarrow.struct.html) | Registro con campos con nombre y tipo. | Dirección con calle, ciudad y código postal. |

Donde `<campos>` es una lista de tuplas `(nombre, tipo)`.

**Ejemplos:**

* La siguiente celda crea un *array* de listas con las cantidades de los artículos de tres pedidos.

In [ ]:
pedidos = pa.array([[1, 2, 3], [4, 5], [6, 7, 8, 9]], type=pa.list_(pa.int32()))
pedidos

* La siguiente celda obtiene el número de artículos de cada pedido con `pc.list_value_length()`.

In [ ]:
pc.list_value_length(pedidos)

* La siguiente celda define un tipo `struct` con los campos `ciudad` y `region`, y crea un *array* con ese tipo.

In [ ]:
tipo_ubicacion = pa.struct([('ciudad', pa.string()), ('region', pa.string())])

ubicaciones = pa.array(
    [
        {'ciudad': 'Monterrey', 'region': 'Norte'},
        {'ciudad': 'Mérida', 'region': 'Sureste'},
        {'ciudad': 'Puebla', 'region': 'Centro'},
    ],
    type=tipo_ubicacion,
)
ubicaciones

* La siguiente celda extrae el campo `ciudad` de cada elemento con `pc.struct_field()`.

In [ ]:
pc.struct_field(ubicaciones, 'ciudad')

## *ChunkedArray* y *RecordBatch*.

Un `pa.Array` ocupa un solo bloque contiguo de memoria. Cuando los datos llegan por partes (por ejemplo, al leer un archivo grande en bloques), copiarlos a un único bloque sería costoso. Para eso existen dos estructuras:

* [`pa.ChunkedArray`](https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html): una columna formada por varios *arrays* (*chunks*) del mismo tipo. Se comporta como un solo *array*.
* [`pa.RecordBatch`](https://arrow.apache.org/docs/python/generated/pyarrow.RecordBatch.html): un bloque de renglones, formado por varios *arrays* de la misma longitud con un *schema* común.

Una tabla de *PyArrow* (`pa.Table`, que se estudia a continuación) es una colección de *record batches*; por eso sus columnas son siempre de tipo `ChunkedArray`.

```
pa.chunked_array([<array_1>, <array_2>, ...])
```

Donde cada `<array_i>` es un *array* o una lista con datos del mismo tipo.

**Ejemplos:**

* La siguiente celda crea un `ChunkedArray` con las ventas de tres días, cada una en un *chunk*.

In [ ]:
ventas_dias = pa.chunked_array([[100, 150, 120], [200, 250], [75, 95, 110, 130]])
ventas_dias

* La siguiente celda muestra el número de *chunks*, la longitud total y la suma. Las funciones de `pyarrow.compute` operan sobre todos los *chunks* como si fueran un solo *array*.

In [ ]:
print(f"Chunks: {ventas_dias.num_chunks}")
print(f"Longitud: {len(ventas_dias)}")
print(f"Suma: {pc.sum(ventas_dias).as_py()}")

* La siguiente celda une los *chunks* en un solo `pa.Array` con `combine_chunks()`. Esta operación sí copia los datos.

In [ ]:
ventas_dias.combine_chunks()

## *Tables* y *schemas*.

Una *table* de *PyArrow* (`pa.Table`) es un conjunto de columnas con nombre y del mismo largo; es el equivalente de un *dataframe*. Su estructura (nombres y tipos de las columnas) se describe con un ***schema*** (`pa.Schema`).

Las formas más comunes de crear una *table* son:

| Función | Origen |
| :--- | :--- |
| [`pa.table()`](https://arrow.apache.org/docs/python/generated/pyarrow.table.html) | Un diccionario de listas o *arrays*. |
| [`pa.Table.from_pandas()`](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.from_pandas) | Un *dataframe* de *Pandas*. |
| [`pyarrow.csv.read_csv()`](https://arrow.apache.org/docs/python/generated/pyarrow.csv.read_csv.html) | Un archivo *CSV*. |
| [`pyarrow.parquet.read_table()`](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.read_table.html) | Un archivo *Parquet*. |

Una *table* es inmutable, como los *arrays*: métodos como `filter()`, `select()`, `sort_by()` o `append_column()` regresan una *table* nueva.

**Ejemplos:**

* La siguiente celda lee `data/ventas.csv` con `pyarrow.csv.read_csv()`. La lectura es multihilo y no pasa por *Pandas*; *PyArrow* infiere los tipos de las columnas, incluida la fecha y hora.

In [ ]:
import pyarrow.csv as pv

ventas = pv.read_csv(ruta_ventas)
print(f"Renglones: {ventas.num_rows:,} · Columnas: {ventas.num_columns}")

* La siguiente celda muestra el *schema* inferido.

In [ ]:
ventas.schema

* La siguiente celda muestra los primeros tres renglones con `slice()`. Cada columna se despliega como un `ChunkedArray`.

In [ ]:
ventas.slice(0, 3)

* La siguiente celda selecciona una columna por nombre; el resultado es un `ChunkedArray`.

In [ ]:
ventas['categoria']

### Operaciones sobre *tables*.

* La siguiente celda agrega la columna `importe` (cantidad por precio unitario) con `append_column()` y selecciona sólo algunas columnas con `select()`.

In [ ]:
importe = pc.multiply(ventas['cantidad'], ventas['precio_unitario'])
ventas = ventas.append_column('importe', importe)

ventas.select(['fecha_hora', 'producto', 'cantidad', 'precio_unitario', 'importe']).slice(0, 5)

* La siguiente celda filtra las ventas en línea de la categoría `Electrónica`. `pc.field()` hace referencia a una columna dentro de una expresión de filtro.

In [ ]:
electronica_en_linea = ventas.filter(
    (pc.field('canal') == 'en línea') & (pc.field('categoria') == 'Electrónica')
)
electronica_en_linea.num_rows

* La siguiente celda agrupa por categoría con `group_by()` y calcula el importe total y el número de ventas con `aggregate()`. Las columnas resultantes se nombran `<columna>_<función>`. Después ordena el resultado con `sort_by()` y redondea el importe total con `set_column()`, que reemplaza una columna por posición.

In [ ]:
por_categoria = (
    ventas
    .group_by('categoria')
    .aggregate([('importe', 'sum'), ('venta_id', 'count')])
    .sort_by([('importe_sum', 'descending')])
)
por_categoria.set_column(1, 'importe_sum', pc.round(por_categoria['importe_sum'], 2))

### *Schemas* explícitos.

Definir el *schema* de antemano garantiza que los datos tengan los tipos esperados. Si un valor no es compatible con su tipo, *PyArrow* desencadena una excepción de inmediato en lugar de convertirlo en silencio.

```
pa.schema([
    (<nombre_1>, <tipo_1>),
    (<nombre_2>, <tipo_2>),
    ...
])
```

Donde cada `<nombre_i>` es el nombre de una columna y `<tipo_i>` su tipo de *Arrow*.

[Documentación de `pa.schema()`](https://arrow.apache.org/docs/python/generated/pyarrow.schema.html)

* La siguiente celda define un *schema* para el catálogo de tiendas.

In [ ]:
schema_tiendas = pa.schema([
    ('tienda_id', pa.int16()),
    ('nombre', pa.string()),
    ('region', pa.string()),
    ('metros_cuadrados', pa.int32()),
])
schema_tiendas

* La siguiente celda crea una *table* que cumple con el *schema*.

In [ ]:
pa.table(
    {
        'tienda_id': [1, 2],
        'nombre': ['Centro Histórico', 'Polanco'],
        'region': ['Centro', 'Centro'],
        'metros_cuadrados': [850, 1200],
    },
    schema=schema_tiendas,
)

* La siguiente celda intenta crear una *table* con texto en la columna `metros_cuadrados`, que debe ser entera. Se desencadenará una excepción de tipo `ArrowInvalid`.

In [ ]:
pa.table(
    {
        'tienda_id': [1, 2],
        'nombre': ['Centro Histórico', 'Polanco'],
        'region': ['Centro', 'Centro'],
        'metros_cuadrados': ['850', 'mil doscientos'],
    },
    schema=schema_tiendas,
)

* Un *schema* también puede aplicarse al leer un *CSV*. La siguiente celda lee `data/tiendas.csv` indicando los tipos de algunas columnas con `ConvertOptions`; las demás se infieren.

In [ ]:
opciones = pv.ConvertOptions(column_types={'tienda_id': pa.int16(), 'fecha_apertura': pa.date32()})
tiendas = pv.read_csv(os.path.join('data', 'tiendas.csv'), convert_options=opciones)
tiendas.schema

## Formatos de archivo.

*PyArrow* lee y escribe varios formatos. Los más relevantes para el análisis de datos son:

| Formato | Tipo | Características |
| :--- | :--- | :--- |
| *CSV* | Texto, por renglones | Universal y legible, pero no guarda tipos de datos, ocupa más espacio y es lento de leer. |
| [*Parquet*](https://parquet.apache.org/) | Binario, columnar | Comprimido, guarda el *schema* y estadísticas por bloque; permite leer sólo algunas columnas o renglones. Es el estándar para almacenar y compartir datos analíticos. |
| [*Feather* / *Arrow IPC*](https://arrow.apache.org/docs/python/feather.html) | Binario, columnar | Es el formato en memoria de *Arrow* escrito a disco: se lee casi sin procesamiento. Útil para archivos temporales e intercambio rápido entre procesos. |

Existen otros formatos columnares, como *ORC*, pero *Parquet* es el más adoptado.

### *Parquet*.

Un archivo *Parquet* divide los datos en grupos de renglones (*row groups*) y, dentro de cada grupo, guarda cada columna por separado junto con estadísticas como el mínimo y el máximo. Gracias a esto, una lectura puede omitir las columnas que no necesita y los grupos cuyos valores no cumplen un filtro.

| Función | Descripción |
| :--- | :--- |
| [`pq.write_table(<table>, <ruta>, compression=<algoritmo>)`](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.write_table.html) | Escribe una *table*; `<algoritmo>` puede ser `'snappy'` (por omisión), `'zstd'` o `'gzip'`, entre otros. |
| [`pq.read_table(<ruta>, columns=<lista>, filters=<filtros>)`](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.read_table.html) | Lee el archivo completo o sólo las columnas y renglones indicados. |
| [`pq.ParquetFile(<ruta>)`](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.ParquetFile.html) | Da acceso a los metadatos sin leer los datos. |

* La siguiente celda escribe la *table* `ventas` en formato *Parquet* con compresión `zstd`.

In [ ]:
import pyarrow.parquet as pq

ruta_parquet = os.path.join(dir_tmp, 'ventas_pyarrow.parquet')
pq.write_table(ventas, ruta_parquet, compression='zstd')

* La siguiente celda consulta los metadatos del archivo sin cargar los datos.

In [ ]:
archivo = pq.ParquetFile(ruta_parquet)
print(f"Renglones: {archivo.metadata.num_rows:,}")
print(f"Grupos de renglones: {archivo.metadata.num_row_groups}")
archivo.schema_arrow

* La siguiente celda lee sólo tres columnas y sólo los renglones de la categoría `Hogar`. Las columnas no solicitadas no se leen del disco.

In [ ]:
hogar = pq.read_table(
    ruta_parquet,
    columns=['fecha_hora', 'producto', 'importe'],
    filters=[('categoria', '==', 'Hogar')],
)
print(f"Renglones: {hogar.num_rows:,}")
hogar.slice(0, 3)

### *Feather* (*Arrow IPC*).

| Función | Descripción |
| :--- | :--- |
| [`feather.write_feather(<table>, <ruta>)`](https://arrow.apache.org/docs/python/generated/pyarrow.feather.write_feather.html) | Escribe una *table* en formato *Feather*. |
| [`feather.read_table(<ruta>, memory_map=<booleano>)`](https://arrow.apache.org/docs/python/generated/pyarrow.feather.read_table.html) | Lee el archivo; con `memory_map=True` el sistema operativo carga los datos sólo cuando se usan. |

* La siguiente celda escribe la *table* `ventas` en formato *Feather* y la lee de nuevo.

In [ ]:
import pyarrow.feather as feather

ruta_feather = os.path.join(dir_tmp, 'ventas_pyarrow.feather')
feather.write_feather(ventas, ruta_feather)

ventas_feather = feather.read_table(ruta_feather, memory_map=True)
ventas_feather.equals(ventas)

* La siguiente celda escribe también un *CSV* y compara el tamaño en disco de los tres formatos.

In [ ]:
ruta_csv = os.path.join(dir_tmp, 'ventas_pyarrow.csv')
pv.write_csv(ventas, ruta_csv)

for ruta in [ruta_csv, ruta_parquet, ruta_feather]:
    print(f"{os.path.basename(ruta):24s} {os.path.getsize(ruta) / 1024:8.0f} KB")

### Conjuntos de datos particionados.

Cuando los datos crecen, es común guardarlos en varios archivos organizados en carpetas según el valor de una o más columnas. A esto se le llama **particionar**. Con el esquema de nombres de *Hive* (`columna=valor`), una consulta que filtra por esa columna sólo abre las carpetas necesarias.

```
data_tmp/ventas_por_anio_pyarrow/
├── anio=2024/
│   └── part-0.parquet
└── anio=2025/
    └── part-0.parquet
```

El módulo [`pyarrow.dataset`](https://arrow.apache.org/docs/python/dataset.html) escribe y lee estos conjuntos de archivos como si fueran una sola tabla.

* La siguiente celda agrega la columna `anio`, extraída de `fecha_hora` con `pc.year()`, y escribe la *table* particionada por año.

In [ ]:
import pyarrow.dataset as ds

ventas_anio = ventas.append_column('anio', pc.year(ventas['fecha_hora']))

ruta_particiones = os.path.join(dir_tmp, 'ventas_por_anio_pyarrow')
ds.write_dataset(
    ventas_anio,
    ruta_particiones,
    format='parquet',
    partitioning=['anio'],
    partitioning_flavor='hive',
    existing_data_behavior='delete_matching',
)
sorted(os.listdir(ruta_particiones))

* La siguiente celda abre el conjunto de datos y lee sólo las ventas de 2025; el archivo de 2024 no se abre. La columna `anio` se reconstruye a partir del nombre de la carpeta.

In [ ]:
conjunto = ds.dataset(ruta_particiones, format='parquet', partitioning='hive')
ventas_2025 = conjunto.to_table(filter=pc.field('anio') == 2025)
ventas_2025.num_rows

## *Arrow* en *Pandas* 3.

En *Pandas* 3, las columnas de texto usan por omisión el tipo `str`, que guarda los datos en un *array* de *Arrow*. Los demás tipos siguen usando *NumPy*, pero cualquier columna puede usar *Arrow* de forma explícita.

| Mecanismo | Descripción |
| :--- | :--- |
| Tipo `str` | Tipo por omisión del texto en *Pandas* 3; usa *Arrow* si *PyArrow* está instalado. |
| [`pd.ArrowDtype(<tipo>)`](https://pandas.pydata.org/docs/reference/api/pandas.ArrowDtype.html) | Usa cualquier tipo de *Arrow* en una columna, por ejemplo `pd.ArrowDtype(pa.int32())`. |
| `dtype_backend="pyarrow"` | Argumento de `pd.read_csv()`, `pd.read_parquet()` y otras funciones de lectura para que todas las columnas usen tipos de *Arrow*. |
| `pa.Table.from_pandas()` / `table.to_pandas()` | Convierten entre *table* y *dataframe*. Por lo general hacen una copia. |

**Ejemplos:**

* La siguiente celda compara la memoria de un millón de cadenas guardadas con el tipo `str` de *Pandas* 3 y con el tipo `object` de versiones anteriores.

In [ ]:
textos = pd.Series(['Electrónica', 'Hogar', 'Papelería', 'Deportes'] * 250_000)
print(f"Tipo por omisión: {textos.dtype} (almacenamiento: {textos.dtype.storage})")

mem_str = textos.memory_usage(deep=True) / 1024**2
mem_object = textos.astype(object).memory_usage(deep=True) / 1024**2
print(f"str:    {mem_str:6.1f} MB")
print(f"object: {mem_object:6.1f} MB")

* La siguiente celda lee `data/ventas.csv` con *Pandas* usando `dtype_backend="pyarrow"`. Las columnas leídas usan tipos de *Arrow* (marcados como `[pyarrow]`), en los que los enteros pueden tener nulos sin convertirse a flotantes. La fecha, que se convierte con `parse_dates`, conserva el tipo `datetime64` de *Pandas*.

In [ ]:
ventas_pd = pd.read_csv(ruta_ventas, dtype_backend='pyarrow', parse_dates=['fecha_hora'])
ventas_pd.dtypes

* La siguiente celda convierte la *table* `tiendas` a un *dataframe* de *Pandas* con `to_pandas()`. El texto pasa al tipo `str`, pero la columna de fechas (`date32`) se convierte en objetos `datetime.date` de *Python*, de tipo `object`.

In [ ]:
tiendas.to_pandas().dtypes

* Con el argumento `types_mapper=pd.ArrowDtype`, todas las columnas conservan su tipo de *Arrow*, incluida la fecha.

In [ ]:
tiendas.to_pandas(types_mapper=pd.ArrowDtype).dtypes

## *Arrow* en *Polars*.

*Polars*, que se estudia a partir del siguiente *notebook*, usa el formato de *Arrow* como representación interna. Por eso la conversión entre una *table* de *PyArrow* y un *dataframe* de *Polars* normalmente no copia los datos.

| Función o método | Descripción |
| :--- | :--- |
| [`pl.from_arrow(<table>)`](https://docs.pola.rs/api/python/stable/reference/api/polars.from_arrow.html) | Convierte una *table* de *PyArrow* en un *dataframe* de *Polars*. |
| [`df.to_arrow()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_arrow.html) | Convierte un *dataframe* de *Polars* en una *table* de *PyArrow*. |

Además, *Polars* lee directamente los archivos *Parquet* y *Feather* escritos con *PyArrow*, y viceversa.

**Ejemplos:**

* La siguiente celda convierte la *table* `ventas` a un *dataframe* de *Polars*.

In [ ]:
import polars as pl

ventas_pl = pl.from_arrow(ventas)
ventas_pl.head(3)

* La siguiente celda lee con *Polars* el archivo *Parquet* que se escribió con *PyArrow*.

In [ ]:
pl.read_parquet(ruta_parquet).shape

* La siguiente celda convierte el *dataframe* de vuelta a una *table* de *PyArrow* con `to_arrow()` y muestra su *schema*. Hay dos diferencias con el original: el texto pasa a `large_string` (la variante de *Arrow* que usa *Polars*) y la fecha pasa de segundos a milisegundos, porque *Polars* no maneja resolución de segundos.

In [ ]:
ventas_de_polars = ventas_pl.to_arrow()
ventas_de_polars.schema

* Los valores, sin embargo, son los mismos. La siguiente celda convierte la *table* al *schema* original con `cast()` y la compara con `ventas`.

In [ ]:
ventas_de_polars.cast(ventas.schema).equals(ventas)

## Resumen.

| Concepto | Descripción |
| :--- | :--- |
| ***Apache Arrow*** | Formato estándar y columnar para datos tabulares en memoria, compartido por muchas herramientas. |
| ***Array*** | Secuencia inmutable de valores de un mismo tipo, con soporte de nulos para cualquier tipo. |
| **`pyarrow.compute`** | Funciones de cálculo sobre *arrays*: aritmética, agregaciones, filtros y texto. |
| **Tipos anidados** | `list_` y `struct` para datos semiestructurados. |
| ***ChunkedArray* y *RecordBatch*** | Columnas y bloques de renglones formados por partes, sin copiar datos. |
| ***Table* y *schema*** | Tabla de columnas con nombre y tipos; el *schema* valida los datos. |
| ***Parquet*** | Formato columnar comprimido para almacenar datos; permite leer sólo columnas y renglones necesarios. |
| ***Feather*** | El formato de *Arrow* en disco, para intercambio rápido. |
| ***Pandas* 3 y *Polars*** | *Pandas* usa *Arrow* para el texto y opcionalmente para cualquier columna; *Polars* lo usa para todo. |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>